In [1]:
!pip install fairlearn

import numpy as np, pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

df = fetch_openml("credit-g", version=1, as_frame=True).frame
y = (df["class"] == "bad").astype(int)        # 1 = predicted to default
X = df.drop(columns="class")

# sensitive attributes
sex = X["personal_status"].str.startswith("female").map({True: "female", False: "male"})
age_group = pd.Series(np.where(X["age"] < 25, "under 25", "25 and over"), index=X.index)

(X_tr, X_te, y_tr, y_te,
 sex_tr, sex_te, age_tr, age_te) = train_test_split(
    X, y, sex, age_group, test_size=0.3, stratify=y, random_state=42)

cat_cols = X.select_dtypes("category").columns
num_cols = X.select_dtypes(exclude="category").columns
prep = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ("num", StandardScaler(), num_cols)])
Xtr = prep.fit_transform(X_tr)
Xte = prep.transform(X_te)

lr = LogisticRegression(max_iter=1000).fit(Xtr, y_tr)
pred = lr.predict(Xte)
print("Accuracy:", accuracy_score(y_te, pred))
print("Recall (bad risks caught):", recall_score(y_te, pred))
print("ROC-AUC:", roc_auc_score(y_te, lr.predict_proba(Xte)[:, 1]))

Accuracy: 0.7833333333333333
Recall (bad risks caught): 0.5333333333333333
ROC-AUC: 0.8010582010582011


In [2]:
from fairlearn.metrics import (MetricFrame, selection_rate, false_positive_rate,
    false_negative_rate, demographic_parity_difference, equalized_odds_difference)

# group sizes in the test set (important for interpreting noisy results)
print(age_te.value_counts(), "\n")
print(sex_te.value_counts(), "\n")

def audit(y_true, y_pred, sensitive, name):
    mf = MetricFrame(
        metrics={"flagged_high_risk": selection_rate,
                 "false_positive_rate": false_positive_rate,
                 "false_negative_rate": false_negative_rate},
        y_true=y_true, y_pred=y_pred, sensitive_features=sensitive)
    print(f"--- {name} ---")
    print(mf.by_group.round(3))
    print("Demographic parity diff:", round(demographic_parity_difference(y_true, y_pred, sensitive_features=sensitive), 3))
    print("Equalized odds diff:", round(equalized_odds_difference(y_true, y_pred, sensitive_features=sensitive), 3), "\n")

audit(y_te, pred, age_te, "Baseline by age")
audit(y_te, pred, sex_te, "Baseline by sex")

25 and over    241
under 25        59
Name: count, dtype: int64 

personal_status
male      199
female    101
Name: count, dtype: int64 

--- Baseline by age ---
                     flagged_high_risk  false_positive_rate  \
sensitive_feature_0                                           
25 and over                      0.212                0.089   
under 25                         0.339                0.226   

                     false_negative_rate  
sensitive_feature_0                       
25 and over                        0.435  
under 25                           0.536  
Demographic parity diff: 0.127
Equalized odds diff: 0.136 

--- Baseline by sex ---
                 flagged_high_risk  false_positive_rate  false_negative_rate
personal_status                                                             
female                       0.297                0.136                0.400
male                         0.206                0.097                0.509
Demographic parity di

In [3]:
from fairlearn.postprocessing import ThresholdOptimizer

results = []
for name, s_tr, s_te in [("age", age_tr, age_te), ("sex", sex_tr, sex_te)]:
    to = ThresholdOptimizer(estimator=lr, constraints="equalized_odds",
                            predict_method="predict_proba", prefit=True)
    to.fit(Xtr, y_tr, sensitive_features=s_tr)
    pred_fair = to.predict(Xte, sensitive_features=s_te, random_state=42)

    audit(y_te, pred_fair, s_te, f"After mitigation by {name}")
    results.append({
        "attribute": name,
        "acc_before": accuracy_score(y_te, pred),
        "acc_after": accuracy_score(y_te, pred_fair),
        "eq_odds_diff_before": equalized_odds_difference(y_te, pred, sensitive_features=s_te),
        "eq_odds_diff_after": equalized_odds_difference(y_te, pred_fair, sensitive_features=s_te),
        "recall_before": recall_score(y_te, pred),
        "recall_after": recall_score(y_te, pred_fair),
    })

print(pd.DataFrame(results).round(3))

--- After mitigation by age ---
                     flagged_high_risk  false_positive_rate  \
sensitive_feature_0                                           
25 and over                      0.095                0.034   
under 25                         0.153                0.129   

                     false_negative_rate  
sensitive_feature_0                       
25 and over                        0.726  
under 25                           0.821  
Demographic parity diff: 0.057
Equalized odds diff: 0.096 

--- After mitigation by sex ---
                 flagged_high_risk  false_positive_rate  false_negative_rate
personal_status                                                             
female                       0.188                0.091                0.629
male                         0.206                0.118                0.564
Demographic parity diff: 0.018
Equalized odds diff: 0.065 

  attribute  acc_before  acc_after  eq_odds_diff_before  eq_odds_diff_after  \
0   

In [4]:
drop_cols = ["personal_status", "age"]
X2_tr = X_tr.drop(columns=drop_cols)
X2_te = X_te.drop(columns=drop_cols)

cat2 = X2_tr.select_dtypes("category").columns
num2 = X2_tr.select_dtypes(exclude="category").columns
prep2 = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat2),
    ("num", StandardScaler(), num2)])
X2tr = prep2.fit_transform(X2_tr)
X2te = prep2.transform(X2_te)

lr2 = LogisticRegression(max_iter=1000).fit(X2tr, y_tr)
pred2 = lr2.predict(X2te)

print("Accuracy:", accuracy_score(y_te, pred2))
print("Recall:", recall_score(y_te, pred2))
print("ROC-AUC:", roc_auc_score(y_te, lr2.predict_proba(X2te)[:, 1]), "\n")

audit(y_te, pred2, age_te, "Sensitive columns removed, by age")
audit(y_te, pred2, sex_te, "Sensitive columns removed, by sex")

Accuracy: 0.78
Recall: 0.5222222222222223
ROC-AUC: 0.7934920634920635 

--- Sensitive columns removed, by age ---
                     flagged_high_risk  false_positive_rate  \
sensitive_feature_0                                           
25 and over                      0.207                0.089   
under 25                         0.339                0.226   

                     false_negative_rate  
sensitive_feature_0                       
25 and over                        0.452  
under 25                           0.536  
Demographic parity diff: 0.132
Equalized odds diff: 0.136 

--- Sensitive columns removed, by sex ---
                 flagged_high_risk  false_positive_rate  false_negative_rate
personal_status                                                             
female                       0.248                0.121                0.514
male                         0.226                0.104                0.455
Demographic parity diff: 0.021
Equalized odds diff: